# EA1 — Diseño e implementación de una base de datos analítica

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | *063* |
| **Integrantes** | *Jorge Andres Ocampo Suarez* |
| **Caso de estudio** | *(Wanderbricks u otro)* |
| **Fecha de entrega** | domingo 23 de agosto |
| **🎥 Enlace al video** | *(pegar aquí — 6 a 9 minutos, mínimo 3 minutos por integrante)* |

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta a la propia.
> Un enlace inaccesible se califica como no entregado.

---
## 1. Contexto y problema

*¿Qué se quiere resolver y por qué importa? Máximo tres párrafos.
Debe quedar claro qué preguntas del negocio deberá responder esta base de datos.*

**Preguntas de negocio:**
- ¿Qué propiedades y anfitriones generan más ingresos, y qué características comparten?
- ¿Qué tan bien se está convirtiendo la navegación en reservas, y dónde se pierden los usuarios en el camino?
- ¿El precio y el rating están correlacionados, o hay propiedades caras con mal desempeño y/o baratas con buen desempeño?

**Contexto:**
Wanderbricks es un marketplace de alquiler vacacional que conecta anfitriones con viajeros, y como todo marketplace, su negocio depende de tres cosas: que las propiedades correctas generen ingresos, que los usuarios que navegan efectivamente reserven, y que el precio se corresponda con la calidad percibida. Sin una base de datos analítica que integre reservas, pagos, reseñas y navegación en un mismo lugar, estas preguntas quedan dispersas en silos operativos y son difíciles de responder de forma consistente.

Esta base de datos analítica debe apoyar decisiones como priorizar qué propiedades y anfitriones promover, identificar en qué punto del recorrido de navegación se pierden los usuarios antes de reservar, y evaluar si la estrategia de precios está alineada con la satisfacción de los huéspedes.

el notebook busca responder: (1) qué propiedades y anfitriones generan más ingresos y qué las diferencia, (2) qué tan efectivo es el funnel de conversión desde la navegación hasta la reserva completada, y (3) si existe relación entre precio por noche y rating de las reseñas.

---
## 2. Descripción de los datos

*Volumen, variedad, tipos, calidad observada y relaciones entre tablas.
Esta descripción es la base de la decisión de diseño de la sección 3: sin ella,
cualquier justificación queda en el aire.*

In [0]:
# Exploración inicial del caso

display(spark.sql("SHOW TABLES IN samples.wanderbricks"))


In [0]:
# Esquema y muestra de cada tabla
tablas = [r[1] for r in spark.sql("SHOW TABLES IN samples.wanderbricks").collect()]
for t in tablas:
    df = spark.table(f"samples.wanderbricks.{t}")
    print(f"--- {t} ---")
    df.printSchema()
    df.limit(5).display()

In [0]:
# Conteo de filas por tabla
for t in [r[1] for r in spark.sql("SHOW TABLES IN samples.wanderbricks").collect()]:
    print(f"{t:30s} {spark.table(f'samples.wanderbricks.{t}').count():>12,}")

In [0]:
# Calidad: nulos por columna en una tabla clave
from pyspark.sql import functions as F
df = spark.table("samples.wanderbricks.bookings")
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).display()

In [0]:
# Relaciones: ¿hay reservas huérfanas (sin propiedad válida)?
bookings = spark.table("samples.wanderbricks.bookings")
properties = spark.table("samples.wanderbricks.properties")
huerfanas = bookings.join(properties, "property_id", "left_anti")
print("reservas sin propiedad:", huerfanas.count())

**TODO: describir las tablas que van a usar (esquema, tipos, nulos, cardinalidades)**

El dataset tiene 16 tablas con volúmenes muy dispares: desde catálogos pequeños como countries (168) o destinations (42), hasta tablas de eventos de alto volumen como page_views (500,000) y clickstream (100,000). Las tablas transaccionales núcleo son bookings (72,247), payments (49,638) y reviews (99,793) — la diferencia entre reservas y pagos (72,247 vs 49,638) sugiere que no todas las reservas llegan a pago completado (canceladas, pendientes), booking_updates (83,068) supera a bookings, lo que indica que cada reserva pasa por varios estados a lo largo de su ciclo de vida.

En calidad, bookings no presenta valores nulos en ninguna de sus columnas, y la verificación de integridad referencial contra properties no arrojó reservas huérfanas

---
## 3. Decisiones de diseño y justificación

*Comparar al menos tres paradigmas —relacional, NoSQL (documental / clave-valor / columnar)
y lakehouse— **atando cada criterio a los datos descritos arriba**. Las ventajas genéricas
copiadas de un manual no cuentan.*

| Criterio del caso | Relacional | NoSQL | Lakehouse | Decisión |
|---|---|---|---|---|
| Integridad transaccional en bookings/payments (sin nulos, sin huérfanas contra properties) | Fuerte — constraints e integridad referencial nativasa | Débil — sin FKs nativos, la integridad quedaría a cargo de la aplicación | Fuerte — Delta soporta constraints y ACID, y permite joins con Spark SQL igual que un relacional | Lakehouse |
| Volumen de eventos (page_views 500K, clickstream 100K) | Limitado — un motor relacional tradicional no está pensado para escribir/leer cientos de miles de eventos de forma eficiente | Fuerte en escritura de eventos, pero débil para agregarlos junto a las tablas transaccionales | Fuerte — escala horizontalmente y permite unir eventos con bookings/properties en el mismo motor | Lakehouse |
| Historial de estados por reserva (booking_updates: 83,068 registros para 72,247 bookings, ~1.15 actualizaciones por reserva) | Posible, pero requiere una tabla de auditoría aparte gestionada manualmente | No ofrece versionado nativo de una tabla completa | Nativo — DESCRIBE HISTORY y time travel dan trazabilidad de cambios sin tabla de auditoría adicional | Lakehouse |
| Escalabilidad futura (empleados internos + usuarios ya suman ~197K registros) | Escala vertical, se vuelve costosa a ese tamaño | Escala horizontal pero sacrificando consistencia | Escala horizontal manteniendo ACID | Lakehouse |

**Referencias (APA 7):**

---
## 4. Implementación
### 4.1 Catálogo, esquema y volumen

In [0]:
CATALOGO = "bigdata_grupo63"   # TODO: reemplazar NN
ESQUEMA  = "wanderbricks"
VOLUMEN  = "datos_crudos"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {CATALOGO}.{ESQUEMA}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN}")

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {ESQUEMA}")
print(f"Trabajando en {CATALOGO}.{ESQUEMA}")

### 4.2 Capa bronce — ingesta de datos crudos

In [0]:
# TODO: ingerir las tablas del caso a la capa bronce, con esquema explícito
from pyspark.sql import functions as F

tablas = [r[1] for r in spark.sql("SHOW TABLES IN samples.wanderbricks").collect()]

for t in tablas:
    (spark.table(f"samples.wanderbricks.{t}")
        .withColumn("_ingested_at", F.current_timestamp())
        .write.format("delta").mode("overwrite")
        .saveAsTable(f"{CATALOGO}.{ESQUEMA}.bronze_{t}"))

print("Tablas bronce creadas:", [f"bronze_{t}" for t in tablas])

Se ingirieron las 16 tablas del catálogo samples.wanderbricks sin ningún filtro ni transformación, replicando el principio de la capa bronce: preservar los datos exactamente como llegan de la fuente, para poder reprocesar desde cero si más adelante se detecta un error en la lógica de limpieza de la capa plata. A cada tabla se le agregó únicamente la columna _ingested_at, que registra el momento en que se ejecutó la ingesta, esto da trazabilidad sobre cuándo entró cada versión de los datos crudos al lakehouse, sin alterar ninguno de los valores originales.

No se aplicó limpieza, casteo de tipos ni eliminación de duplicados en esta etapa: esas decisiones se ejecutan en la capa plata, donde sí importa dejar rastro explícito de qué regla de negocio se aplicó y por qué.

### 4.3 Capa plata — datos limpios y tipados

In [0]:
%sql
-- TODO: limpieza, tipado y reglas de negocio
CREATE OR REPLACE TABLE bigdata_grupo63.wanderbricks.silver_bookings (
    booking_id      STRING NOT NULL,
    user_id         STRING NOT NULL,
    property_id     STRING NOT NULL,
    check_in        DATE,
    check_out       DATE,
    guests_count    INT,
    total_amount    DECIMAL(10,2),
    status          STRING,
    created_at      TIMESTAMP,
    updated_at      TIMESTAMP
) USING DELTA;

INSERT INTO bigdata_grupo63.wanderbricks.silver_bookings
SELECT booking_id, user_id, property_id,
       CAST(check_in AS DATE), CAST(check_out AS DATE),
       CAST(guests_count AS INT),
       CAST(total_amount AS DECIMAL(10,2)),
       status, created_at, updated_at
FROM bigdata_grupo63.wanderbricks.bronze_bookings
WHERE total_amount >= 0
  AND check_out > check_in;

In [0]:
%sql
CREATE OR REPLACE TABLE bigdata_grupo63.wanderbricks.silver_payments AS
SELECT * FROM bigdata_grupo63.wanderbricks.bronze_payments
WHERE booking_id IS NOT NULL;

CREATE OR REPLACE TABLE bigdata_grupo63.wanderbricks.silver_properties AS
SELECT * FROM bigdata_grupo63.wanderbricks.bronze_properties
WHERE property_id IS NOT NULL;

CREATE OR REPLACE TABLE bigdata_grupo63.wanderbricks.silver_reviews AS
SELECT * FROM bigdata_grupo63.wanderbricks.bronze_reviews
WHERE booking_id IS NOT NULL;

Sobre la capa bronce se construyó la capa plata aplicando esquema explícito y reglas de calidad concretas, en lugar de copiar los datos tal como llegaron. En silver_bookings se tiparon explícitamente fechas (DATE), montos (DECIMAL(10,2)) y cantidad de huéspedes (INT) — en bronce estos campos quedan con el tipo genérico de la ingesta cruda —, y se descartaron dos tipos de registros inconsistentes: reservas con total_amount negativo (que no tiene sentido de negocio) y reservas donde la fecha de salida no es posterior a la de entrada. Ninguna de estas dos condiciones apareció en la verificación de nulos, pero se agregan como salvaguarda explícita del modelo.

Para payments y reviews se filtró cualquier registro sin booking_id, ya que ambas tablas solo tienen sentido analítico si pueden vincularse a una reserva concreta; para properties se exigió property_id no nulo por ser la llave que sostiene la mayoría de los joins del notebook

### 4.4 Datos semiestructurados

*Al menos una tabla debe manejar estructuras anidadas (structs o arrays).
El clickstream y las reseñas son los candidatos naturales.*

In [0]:
# TODO: leer y aplanar estructuras anidadas
df = spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_clickstream")
df.printSchema()
df.select("metadata").limit(5).display()

In [0]:
from pyspark.sql import functions as F

# Aplanar el struct 'metadata' con notación de punto
flat = df.select(
    "event", "property_id", "timestamp", "user_id",
    "metadata.device", "metadata.referrer"
)
flat.display()

In [0]:
%sql
CREATE OR REPLACE TABLE bigdata_grupo63.wanderbricks.silver_clickstream AS
SELECT
    event,
    property_id,
    CAST(timestamp AS TIMESTAMP) AS event_timestamp,
    user_id,
    metadata.device AS device,
    metadata.referrer AS referrer
FROM bigdata_grupo63.wanderbricks.bronze_clickstream;

clickstream se eligió como la tabla candidata para trabajar estructuras anidadas: cada evento de navegación  trae un campo metadata de tipo struct con device, desde qué tipo de dispositivo se generó el evento y referrer, el canal de origen, por ejemplo email. 

### 4.5 Propiedades del lakehouse

*Hay que evidenciar las tres: atomicidad, time travel y evolución de esquema.*

In [0]:
%sql
MERGE INTO bigdata_grupo63.wanderbricks.silver_bookings AS target
USING bigdata_grupo63.wanderbricks.bronze_bookings AS source
ON target.booking_id = source.booking_id
WHEN MATCHED THEN UPDATE SET target.status = source.status, target.updated_at = source.updated_at
WHEN NOT MATCHED THEN INSERT *;


Atomicidad: el MERGE sobre silver_bookings actualiza el estado de las reservas existentes e inserta las nuevas en una sola operación transaccional.

In [0]:
%sql
SELECT status, COUNT(*) AS cantidad
FROM bigdata_grupo63.wanderbricks.silver_bookings
GROUP BY status
ORDER BY cantidad DESC;

Consistencia: Delta garantiza que el MERGE no modifique datos inconsistentes. Por ejemplo, si se intenta actualizar el estado de una reserva que no existe, Delta no permitirá la operación.

In [0]:
%sql
DESCRIBE HISTORY bigdata_grupo63.wanderbricks.silver_bookings;

--SELECT * FROM bigdata_grupo63.wanderbricks.silver_bookings VERSION AS OF 0
--LIMIT 10;

Versionado (time travel): DESCRIBE HISTORY deja un registro de cada operación de escritura sobre silver_bookings; el MERGE, la carga inicial, la evolución de esquema, con quién, cuándo y qué tipo de operación se ejecutó. La consulta con VERSION AS OF permite reconstruir el estado de la tabla exactamente antes del MERGE.

In [0]:
# Evolución de esquema con mergeSchema
# TODO

from pyspark.sql import functions as F

nuevo_df = (spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_bookings")
    .withColumn("cancellation_reason", F.lit(None).cast("string")))

(nuevo_df.write.format("delta").mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.silver_bookings"))

Evolución de esquema: se agregó la columna cancellation_reason a silver_bookings sin recrear la tabla ni interrumpir las consultas existentes, usando mergeSchema. Esto demuestra que el modelo puede crecer conforme cambian las necesidades del negocio

### 4.6 Consultas analíticas

*Mínimo cinco, en SQL y en PySpark, que respondan preguntas reales del negocio.
Consultas triviales sin conexión con el problema no puntúan.*

In [0]:
%sql
SELECT p.property_id, p.title, SUM(b.total_amount) AS ingresos_totales
FROM bigdata_grupo63.wanderbricks.silver_bookings b
JOIN bigdata_grupo63.wanderbricks.silver_properties p ON b.property_id = p.property_id
WHERE b.status = 'completed'
GROUP BY p.property_id, p.title
ORDER BY ingresos_totales DESC
LIMIT 10;

In [0]:
bookings = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_bookings")
properties = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_properties")

(bookings.filter(F.col("status") == "completed")
 .join(properties, "property_id")
 .groupBy("property_id", "title")
 .agg(F.sum("total_amount").alias("ingresos_totales"))
 .orderBy(F.desc("ingresos_totales"))
 .limit(10)
 .display())

---
## 5. Resultados

*Qué se obtuvo. Las salidas de las celdas deben quedar visibles en el notebook exportado.*

---
## 6. Conclusiones

*Qué funcionó, qué no funcionó y qué harían distinto si empezaran de nuevo.
Las conclusiones deben derivarse de los resultados mostrados arriba, no de expectativas generales.*

---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| | | |
| | | |
| | | |

**Uso de asistentes de IA:** *(indicar en qué partes se usó el Databricks Assistant u otra
herramienta. Está permitido; lo que se evalúa es que cada integrante pueda explicar
cualquier línea del código en el video.)*

> Este reparto debe coincidir con lo que cada persona demuestra en el video y con el
> historial de commits del repositorio. Las tres fuentes se contrastan al calificar.

---
## 📹 Preguntas obligatorias de sustentación

Cada integrante responde estas tres preguntas en su intervención del video:

1. ¿Por qué eligieron este modelo de datos y qué alternativa descartaron?
2. Muestre una consulta que usted escribió y explique qué hace Spark al ejecutarla.
3. ¿Qué tendrían que cambiar en su diseño si el volumen se multiplicara por cien?

---
## ✅ Antes de entregar

- [ ] El notebook corre completo de arriba abajo sin errores
- [ ] Las siete secciones están diligenciadas, no quedaron textos de plantilla
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] Todos los integrantes aparecen en el video con cámara al presentarse
- [ ] El notebook está confirmado en el repositorio, en la carpeta /ea1
- [ ] El HTML con salidas visibles está subido a Canvas